# 📓 Semana 2 · Dia 2 — Qualidade de dados: 6 dimensões e constraints

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (qualidade de dados) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Regras de qualidade aplicadas no Bronze |

---


## 📖 Teoria — As 6 dimensões de qualidade de dados

| Dimensão | Pergunta | Exemplo de falha |
|---|---|---|
| **Completude** | Todos os valores estão presentes? | `CustomerID` nulo |
| **Unicidade** | Há duplicatas? | mesma nota 2x |
| **Validade** | Valores respeitam o domínio? | `Quantity = -999` |
| **Pontualidade** | Os dados chegaram a tempo? | ingestão atrasada 3h |
| **Precisão** | Os valores são exatos? | preço 10.0 registrado 9.97 |
| **Consistência** | Dados coerentes entre fontes? | país 'BR' vs 'BRA' |

> 🎯 **Dica de prova**: a DEA pede para **classificar** um problema de qualidade na dimensão correta. Ex.: duplicata = unicidade; CPF inválido = validade; campo vazio = completude.


## 📖 Teoria — Enforcement de qualidade no Delta

O Delta Lake permite **constraints** que o motor **impõe na escrita**:

```sql
ALTER TABLE t ADD CONSTRAINT ck_qtd CHECK (Quantity > 0);
ALTER TABLE t ADD CONSTRAINT nn_cliente NOT NULL (CustomerID);
```

Se uma escrita violar a constraint, a transação **falha inteira** (ACID) — nenhuma linha entra. Isso é a diferença entre 'esperar qualidade' e 'garantir qualidade'.


### 💻 Na prática — Profiling do Bronze

Primeiro: conheça o dado. Faça o perfil de completude, unicidade e validade.


In [ ]:
# Profiling: completude e unicidade
from pyspark.sql.functions import col, count, countDistinct, isnan, isnull
df = spark.table("workspace.bronze.vendas_bronze")
df.select([count(isnull(c)).alias(f"nulos_{c}") for c in df.columns]).show()
print("Linhas totais:", df.count())
print("Notas únicas:", df.select(countDistinct("InvoiceNo")).collect()[0][0])

In [ ]:
# Profiling: validade e consistência
display(df.select("Country").distinct().orderBy("Country"))
df.filter("Quantity <= 0 OR UnitPrice <= 0").count()

### 💻 Na prática — Aplicando constraints

Agora travamos as regras no schema — qualquer escrita que viole falha a transação.


In [ ]:
%sql
ALTER TABLE workspace.bronze.vendas_bronze
  ADD CONSTRAINT ck_vendas_quantidade_positiva CHECK (Quantity > 0);
ALTER TABLE workspace.bronze.vendas_bronze
  ADD CONSTRAINT ck_vendas_preco_positivo CHECK (UnitPrice > 0);
SHOW TBLPROPERTIES workspace.bronze.vendas_bronze ("delta.constraints.*")

In [ ]:
# Testar a constraint: essa escrita DEVE falhar
from pyspark.sql import Row
try:
    spark.createDataFrame([Row(InvoiceNo="T", StockCode="X", Description="", Quantity=-1,
                                 InvoiceDate="2024-01-01", UnitPrice=10.0, CustomerID=None,
                                 Country="BR")]) \
        .write.mode("append").saveAsTable("workspace.bronze.vendas_bronze")
    print("ERRO: deveria ter falhado!")
except Exception as e:
    print("Constraint funcionou — escrita bloqueada:", str(e)[:120])

> 🎯 **Dica de prova**: Constraints `CHECK` e `NOT NULL` são garantia de prova (DEA). Memorize a sintaxe e o comportamento: violação → transação inteira falha.


## 🎯 Exercícios de fixação

**1.** Classifique em qual dimensão cai: (a) mesmo cliente com 2 IDs; (b) email inválido; (c) coluna vazia; (d) dado de ontem que chegou hoje.

**2.** Adicione uma constraint que garanta que `UnitPrice >= 0`.

**3.** Por que constraints falham a transação inteira em vez de descartar a linha?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Classificação

(a) unicidade; (b) validade; (c) completude; (d) pontualidade.

**2.** Constraint

```sql
ALTER TABLE workspace.bronze.vendas_bronze ADD CONSTRAINT ck_preco CHECK (UnitPrice >= 0);
```

**3.** Transação inteira

Porque o Delta é ACID: uma escrita que viola constraint fica 'incompleta' — aceitar parcialmente quebraria a atomicidade e a consistência. O produtor deve corrigir o dado na origem.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*